# 03 - Clipped mask schedule 탐색

**학습 목표**: mask probability `t`에서 `masked_count / (n*t)` estimator의 평균과 분산을 측정하고, 후보 구간 중 낮은 분산을 고릅니다. 논문의 neural gradient variance를 재현하는 것은 아닙니다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 라이브러리만 사용합니다.

In [ ]:
import random
import statistics

def draw_estimator(token_count, beta, omega, rng):
    t = rng.uniform(beta, omega)
    masked = sum(rng.random() < t for _ in range(token_count))
    return masked / (token_count * t)

def evaluate_range(token_count, beta, omega, trials=10000, seed=0):
    rng = random.Random(seed)
    values = [draw_estimator(token_count, beta, omega, rng) for _ in range(trials)]
    return statistics.mean(values), statistics.pvariance(values)

candidates = [(0.01, 0.99), (0.10, 0.90), (0.30, 0.80), (0.50, 0.95)]
for block_size in (4, 16, 128):
    print(f'block size={block_size}')
    for beta, omega in candidates:
        mean, variance = evaluate_range(block_size, beta, omega, seed=block_size)
        print(f'  U[{beta:0.2f},{omega:0.2f}] mean={mean:0.3f} variance={variance:0.4f}')


In [ ]:
def select_low_variance_range(block_size):
    scored = []
    for beta, omega in candidates:
        mean, variance = evaluate_range(block_size, beta, omega, seed=99 + block_size)
        # 평균이 1에서 너무 멀면 잘못된/불안정한 후보로 penalty를 줍니다.
        score = variance + 10 * abs(mean - 1)
        scored.append((score, beta, omega, mean, variance))
    return min(scored)

for block_size in (4, 16, 128):
    score, beta, omega, mean, variance = select_low_variance_range(block_size)
    print(f'block={block_size:3d} selected U[{beta},{omega}], variance={variance:0.4f}')
    assert abs(mean - 1) < 0.1

이 estimator에서는 넓은 구간의 작은 `t`가 분산을 크게 만듭니다. 실제 논문의 최적 범위는 model loss와 data distribution에 따라 block size별로 달랐습니다. 따라서 이 toy가 선택한 범위를 논문 hyperparameter로 사용하면 안 됩니다.